# Initial-state injection

Every simulation entry point starts from $|0\dots0\rangle$ by default.  For
time-stepped workloads (step → snapshot → re-seed loops), `initial_state=`
seeds the register with previously computed amplitudes instead — an
$O(2^n)$ copy per step, versus re-simulating from scratch ($O(N^2)$ total
work over $N$ steps) or re-encoding through `SynthesizedStateBlock`
($O(4^n)$ simulation cost per step).

Amplitudes are LSB-indexed (qubit 0 = bit 0 of the index), length
$2^{n}$, unit norm within $10^{-10}$ — rejected otherwise, never
renormalised.  Simulator-only physics: a routed device or a non-Qarp
engine raises `CapabilityError`; on hardware-faithful backends encode the
state as gates (`SynthesizedStateBlock`).

## Block level — `block.statevector(initial_state=...)`

In [ ]:
import numpy as np

from qarp.blocks import SimpleBlock

step = SimpleBlock(2, name="step")
step.ry(0, 0.4)
step.cx(0, 1)
step.rz(1, 0.9)
step.build()

# The statevector output feeds back in unchanged: an N-step loop is N
# single-step simulations, not one N-step re-simulation per snapshot.
psi = np.array([1, 0, 0, 0], dtype=complex)
for k in range(4):
    psi = step.statevector(initial_state=psi)
psi

## Primitive level — `Sampler` / `StateVector` with `QarpEngine`

`initial_state=` on the primitive seeds the **ket** (bra circuits stay
$|0\dots0\rangle$-rooted).  Build once; the seed is mutable between runs.

In [ ]:
import qarp
from qarp.algorithms import Sampler, StateVector
from qarp.engines import QarpEngine
from qarp.operators import QubitOperator

sampler = Sampler(ket=step, n_shots=qarp.EXACT)
energy = StateVector(ket=step, operator=QubitOperator("Z0 Z1"))

engine = QarpEngine()
engine.build([sampler, energy])

psi = np.array([1, 0, 0, 0], dtype=complex)
for k in range(4):
    sampler.initial_state = psi
    energy.initial_state = psi
    dist, ev = engine.run()
    psi = step.statevector(initial_state=psi)  # snapshot for the next step
    print(f"step {k}: <Z0 Z1> = {ev.real:+.6f}   P(00) = {dist.get((0, 0), 0.0):.6f}")

Validation is strict — wrong length or non-unit norm raises `ValueError`
naming the check:

In [ ]:
try:
    step.statevector(initial_state=np.array([0.5, 0.5, 0.5, 0.4]))
except ValueError as e:
    print(e)